# 02 — Chunking & Embeddings

In this notebook we will:
1. Load the parsed transcript from notebook 01
2. Understand what embeddings are and why we need them
3. Build ChromaDB documents with metadata from speaker turns
4. Store everything in a persistent ChromaDB collection

### Key concepts
- **Chunk**: A unit of text that gets its own embedding. We're using speaker turns as chunks — each intervention is one chunk.
- **Embedding**: A numerical vector (array of floats) that captures the *semantic meaning* of text. Similar meanings → similar vectors.
- **Vector store**: A database optimized for storing and searching embeddings by similarity. We use ChromaDB.
- **Metadata**: Structured data attached to each chunk (company, quarter, speaker, role). Enables filtering during retrieval.

## Setup

In [1]:
import json
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions

## Step 1: Load the parsed transcript

We load the JSON file we created in notebook 01.

In [2]:
transcript_path = Path("../data/processed/AAPL_Q1_2025.json")

with open(transcript_path, "r", encoding="utf-8") as f:
    transcript = json.load(f)

turns = transcript["turns"]
print(f"Company: {transcript['company']}")
print(f"Quarter: {transcript['quarter']}")
print(f"Total turns: {len(turns)}")
print(f"Speakers: {transcript['speakers'][:5]}...")

Company: AAPL
Quarter: Q1-2025
Total turns: 75
Speakers: ['Ben Reitzes', 'Mike Ng', 'Richard Kramer', 'Benjamin Bollin', 'Amit Daryanani']...


## Step 2: Understand embeddings

Before storing anything, let's see what an embedding actually looks like.

We'll use `all-MiniLM-L6-v2` from sentence-transformers — a lightweight model that runs on CPU and produces 384-dimensional vectors.

In [3]:
# Create the embedding function that ChromaDB will use
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Let's see what an embedding looks like
sample_texts = [
    "Revenue grew 15% year over year",
    "Sales increased significantly compared to last year",
    "The weather in Tokyo is sunny today",
]

sample_embeddings = embedding_fn(sample_texts)

print(f"Number of embeddings: {len(sample_embeddings)}")
print(f"Dimensions per embedding: {len(sample_embeddings[0])}")
print(f"First 10 values of embedding 1: {sample_embeddings[0][:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of embeddings: 3
Dimensions per embedding: 384
First 10 values of embedding 1: [ 0.03196584  0.00702383  0.02566922 -0.05172473  0.03063454 -0.02880665
 -0.04893331  0.05552837 -0.0352448   0.04485524]


In [4]:
# Let's measure similarity between our sample texts.
# Cosine similarity: 1.0 = identical meaning, 0.0 = unrelated
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_1_2 = cosine_similarity(sample_embeddings[0], sample_embeddings[1])
sim_1_3 = cosine_similarity(sample_embeddings[0], sample_embeddings[2])

print(f"'Revenue grew...' vs 'Sales increased...': {sim_1_2:.3f}")
print(f"'Revenue grew...' vs 'Weather in Tokyo...': {sim_1_3:.3f}")

'Revenue grew...' vs 'Sales increased...': 0.588
'Revenue grew...' vs 'Weather in Tokyo...': 0.011


## Step 3: Build documents and metadata for ChromaDB

ChromaDB needs three parallel lists to store documents:
- `documents`: the text content (list of strings)
- `metadatas`: structured data for each document (list of dicts)
- `ids`: a unique identifier for each document (list of strings)

Each turn from our transcript becomes one document in ChromaDB.

---

Build the three lists from `turns`. Each turn dict has keys: `speaker`, `role`, `section`, `text`.

The transcript-level metadata (`company`, `quarter`, `date`) also needs to be attached to every document — ChromaDB doesn't have a concept of "parent document", so each chunk must carry its own context.

For `ids`, they must be unique strings. Think about what combination of fields would make a good unique identifier.

In [5]:
documents = []
metadatas = []
ids = []

for i, turn in enumerate(turns):
    dict_transcript = {
        'company': transcript['company'], 'quarter': transcript['quarter'], 
        'speaker': turn["speaker"], 'role': turn["role"], 'section': turn["section"] 
    }
    turn_id = f"{transcript['company']}_{transcript['quarter']}_{i}"
    documents.append(turn["text"])
    metadatas.append(dict_transcript)
    ids.append(turn_id)

print(f"Documents: {len(documents)}")
print(f"Metadatas: {len(metadatas)}")
print(f"IDs: {len(ids)}")
print(f"\nSample document (first 100 chars): {documents[0][:100]}...")
print(f"Sample metadata: {metadatas[0]}")
print(f"Sample ID: {ids[0]}")

Documents: 75
Metadatas: 75
IDs: 75

Sample document (first 100 chars): Good afternoon, and welcome to the Apple Q1 fiscal year 2025 earnings conference call. My name is Su...
Sample metadata: {'company': 'AAPL', 'quarter': 'Q1-2025', 'speaker': 'Suhasini Chandramouli', 'role': 'Director, Investor Relations', 'section': 'prepared_remarks'}
Sample ID: AAPL_Q1-2025_0


## Step 4: Store in ChromaDB

Now we create a persistent ChromaDB collection and add our documents.

ChromaDB handles embedding generation automatically when we pass an `embedding_function` to the collection — we don't need to call it manually.

In [6]:
# Create a persistent ChromaDB client (data saved to disk)
client = chromadb.PersistentClient(path="../chroma_db")

# Delete collection if it exists (for re-runs)
try:
    client.delete_collection("earnings_calls")
except Exception:
    pass

# Create the collection with our embedding function
collection = client.create_collection(
    name="earnings_calls",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},  # use cosine similarity
)

print(f"Collection created: {collection.name}")

Collection created: earnings_calls


In [7]:
# Add all documents to the collection
# ChromaDB will automatically generate embeddings using our embedding_fn
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
)

print(f"Added {collection.count()} documents to the collection")

Added 75 documents to the collection


## Summary

In this notebook you learned:
- What embeddings are and how semantic similarity works (cosine similarity)
- How to structure documents with metadata for a vector store
- How ChromaDB stores and queries documents by meaning, not just keywords
- How metadata filtering narrows search results (e.g., only CFO statements)

**Next step:** In notebook 03 we'll explore different retrieval strategies — semantic search, metadata filtering, and combining both.